In [1]:
# ── Cell 1: Imports & Configuration ──────────────────────────────────────────
import requests
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

# ── Only these two variables need to change ───────────────────────────────────
TICKER     = "OKE"
USER_AGENT = "sge42@txstate.edu"   # Required by SEC fair-use policy
# ─────────────────────────────────────────────────────────────────────────────

TODAY       = datetime.today().strftime('%Y-%m-%d')
OUTPUT_FILE = f"{TICKER}_10K_Historical_{TODAY}.xlsx"

In [2]:
# ── Cell 2: Helper Functions ──────────────────────────────────────────────────

HEADERS     = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate", "Host": "data.sec.gov"}
HEADERS_SEC = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate", "Host": "www.sec.gov"}

def fetch_json(url, retries=3):
    h = HEADERS if "data.sec.gov" in url else HEADERS_SEC
    for attempt in range(retries):
        try:
            time.sleep(0.15)
            r = requests.get(url, headers=h, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise e

def to_millions(value, unit):
    if value is None:
        return None
    return round(value / 1_000_000, 2) if unit in ("USD", "shares") else value

def resolve_tag(facts_gaap, tag_candidates, fy_end_set, negate=False):
    """Try tags in ranked order; return ({fy_end: value}, tag_used) for the first hit."""
    for tag in tag_candidates:
        if tag not in facts_gaap:
            continue
        units_dict = facts_gaap[tag].get("units", {})
        unit_key   = next((k for k in units_dict if k in ("USD", "shares", "pure")), None)
        if unit_key is None:
            continue
        annual = [e for e in units_dict[unit_key] if e.get("form") in ("10-K", "10-K/A")]
        fy_map = {}
        for e in annual:
            end = e.get("end")
            if end not in fy_end_set:
                continue
            filed = e.get("filed", "")
            if end not in fy_map or filed > fy_map[end]["filed"]:
                fy_map[end] = {"val": e.get("val"), "unit": unit_key, "filed": filed}
        if not fy_map:
            continue
        result = {}
        for end, info in fy_map.items():
            v = to_millions(info["val"], info["unit"])
            result[end] = (-v if negate and v is not None else v)
        return result, tag
    return {}, None

def build_row(facts_gaap, tag_candidates, fy_ends, negate=False):
    data, tag_used = resolve_tag(facts_gaap, tag_candidates, set(fy_ends), negate=negate)
    return [data.get(fy) for fy in fy_ends], tag_used

In [3]:
# ── Cell 3: CIK Lookup & 10-K Filing Identification ──────────────────────────

tickers_data = fetch_json("https://www.sec.gov/files/company_tickers.json")
ticker_upper = TICKER.upper()
cik_raw = company_name = None

for entry in tickers_data.values():
    if entry["ticker"].upper() == ticker_upper:
        cik_raw      = entry["cik_str"]
        company_name = entry["title"]
        break

if cik_raw is None:
    raise ValueError(f"Ticker '{TICKER}' not found in SEC ticker list")

CIK = str(cik_raw).zfill(10)
print(f"\u2705 Resolved {TICKER} \u2192 {company_name} (CIK: {CIK})")

submissions = fetch_json(f"https://data.sec.gov/submissions/CIK{CIK}.json")
filings     = submissions.get("filings", {}).get("recent", {})

tenk_filings = [
    {"accessionNumber": filings["accessionNumber"][i],
     "filingDate":      filings["filingDate"][i],
     "reportDate":      filings["reportDate"][i],
     "primaryDocument": filings["primaryDocument"][i]}
    for i, form in enumerate(filings.get("form", []))
    if form in ("10-K", "10-K/A")
]

tenk_filings.sort(key=lambda x: x["filingDate"], reverse=True)
tenk_filings = tenk_filings[:5]
tenk_filings.sort(key=lambda x: x["reportDate"])    # oldest → newest

FY_ENDS   = [f["reportDate"] for f in tenk_filings]
FY_LABELS = [f"FY{f['reportDate'][:4]}" for f in tenk_filings]

# Forecast labels start one year after the most recent 10-K
MOST_RECENT_YEAR = int(FY_ENDS[-1][:4])
PROJ_LABELS      = [f"FY{MOST_RECENT_YEAR + i}E" for i in range(1, 4)]

print(f"\u2705 Found {len(tenk_filings)} 10-K filings : {', '.join(FY_LABELS)}")
print(f"\u2705 Forecast columns              : {', '.join(PROJ_LABELS)}")

✅ Resolved OKE → ONEOK INC /NEW/ (CIK: 0001039684)
✅ Found 5 10-K filings : FY2021, FY2022, FY2023, FY2024, FY2025
✅ Forecast columns              : FY2026E, FY2027E, FY2028E


In [4]:
# ── Cell 4: XBRL Company Facts Download ──────────────────────────────────────

facts_data = fetch_json(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json")
facts_gaap = facts_data.get("facts", {}).get("us-gaap", {})
print(f"\u2705 Pulled XBRL company facts (us-gaap concepts: {len(facts_gaap)})")

✅ Pulled XBRL company facts (us-gaap concepts: 565)


In [5]:
# ── Cell 5: Full XBRL Harvest & Auto-Classification ──────────────────────────
import re
import xml.etree.ElementTree as ET
from datetime import date as _date

CFS_SUBSTRINGS = (
    "NetCashProvided", "NetCashUsed", "ProceedsFrom", "PaymentsTo",
    "PaymentsFor", "PaymentsOf", "IncreaseDecreaseIn", "RepaymentsOf",
    "RepurchaseOf", "EffectOfExchangeRate", "CashCashEquivalentsPeriod",
    "CashAndCashEquivalentsPeriodIncrease",
)

PRIORITY_TAGS = {
    "Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax",
    "RevenueFromContractWithCustomerIncludingAssessedTax", "SalesRevenueNet",
    "GrossProfit", "OperatingIncomeLoss", "NetIncomeLoss",
    "EarningsPerShareBasic", "EarningsPerShareDiluted",
    "IncomeTaxExpenseBenefit",
    "InterestAndDividendIncomeOperating", "InterestIncomeExpenseNet",
    "InterestExpense", "NoninterestIncome", "NoninterestExpense",
    "ProvisionForLoanLeaseAndOtherLosses", "ProvisionForLoanAndLeaseLosses",
    "Assets", "AssetsCurrent", "AssetsNoncurrent",
    "Liabilities", "LiabilitiesCurrent",
    "StockholdersEquity",
    "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    "LiabilitiesAndStockholdersEquity",
    "CashAndCashEquivalentsAtCarryingValue",
    "LoansAndLeasesReceivableNetReportedAmount",
    "LoansAndLeasesReceivableGrossCarryingAmount",
    "AllowanceForLoanAndLeaseLosses",
    "Deposits",
    "NetCashProvidedByUsedInOperatingActivities",
    "NetCashProvidedByUsedInInvestingActivities",
    "NetCashProvidedByUsedInFinancingActivities",
}

def _is_annual(start, end):
    try:
        return 330 <= (_date.fromisoformat(end) - _date.fromisoformat(start)).days <= 400
    except:
        return True

def _classify(tag, period_type):
    if period_type == "instant":
        return "BS"
    return "CFS" if any(s in tag for s in CFS_SUBSTRINGS) else "IS"

def harvest_all_facts(facts_gaap, fy_ends):
    fy_set = set(fy_ends)
    results = {}
    for tag, concept in facts_gaap.items():
        label    = concept.get("label") or tag
        units_d  = concept.get("units", {})
        unit_key = next((k for k in units_d if k in ("USD", "shares", "pure")), None)
        if not unit_key:
            continue
        annual = [e for e in units_d[unit_key] if e.get("form") in ("10-K", "10-K/A")]
        if not annual:
            continue
        n_dur       = sum(1 for e in annual if "start" in e)
        period_type = "duration" if n_dur > len(annual) / 2 else "instant"
        fy_map = {}
        for e in annual:
            end = e.get("end")
            if end not in fy_set:
                continue
            if period_type == "duration" and "start" in e:
                if not _is_annual(e["start"], end):
                    continue
            filed = e.get("filed", "")
            if end not in fy_map or filed > fy_map[end]["filed"]:
                fy_map[end] = {"val": e.get("val"), "unit": unit_key, "filed": filed}
        if not fy_map:
            continue
        results[tag] = {
            "label":       label,
            "period_type": period_type,
            "unit":        unit_key,
            "values":      {end: to_millions(d["val"], d["unit"]) for end, d in fy_map.items()},
            "stmt":        _classify(tag, period_type),
            "priority":    tag in PRIORITY_TAGS,
        }
    return results

# ── Fetch presentation order from FilingSummary.xml + R-files ─────────────────
def get_filing_concept_order(cik, accession_number):
    """
    Reads the most recent 10-K's FilingSummary.xml to identify which R-files
    are the IS / BS / CFS statements, then parses each R-file HTML to extract
    the exact row order of us-gaap concepts as filed.
    Returns three dicts {tag: order_index} for IS, BS, CFS.
    """
    int_cik = int(cik)
    accn    = accession_number.replace("-", "")
    base    = f"https://www.sec.gov/Archives/edgar/data/{int_cik}/{accn}"
    hdrs    = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate"}

    try:
        time.sleep(0.2)
        resp = requests.get(f"{base}/FilingSummary.xml", headers=hdrs, timeout=30)
        resp.raise_for_status()
        root = ET.fromstring(resp.content)
    except Exception as e:
        print(f"⚠️  FilingSummary.xml unavailable ({e}); using alphabetical order")
        return {}, {}, {}

    IS_KW  = ("income", "operation", "earning", "comprehensive")
    BS_KW  = ("balance sheet", "financial position")
    CFS_KW = ("cash flow",)

    stmt_rfiles = []
    for report in root.iter("Report"):
        cat  = (report.get("menuCat") or "").lower()
        name = (report.get("longName") or "").lower()
        inst = (report.get("instance") or report.get("htmlFileName") or "")
        if "statement" not in cat:
            continue
        m = re.search(r'[Rr](\d+)\.htm', inst)
        if not m:
            continue
        r_num = int(m.group(1))
        if   any(k in name for k in IS_KW):  stmt_rfiles.append((r_num, "IS"))
        elif any(k in name for k in BS_KW):  stmt_rfiles.append((r_num, "BS"))
        elif any(k in name for k in CFS_KW): stmt_rfiles.append((r_num, "CFS"))

    is_o, bs_o, cfs_o = {}, {}, {}
    order_by_stmt = {"IS": is_o, "BS": bs_o, "CFS": cfs_o}

    for r_num, stmt_type in stmt_rfiles:
        try:
            time.sleep(0.15)
            r = requests.get(f"{base}/R{r_num}.htm", headers=hdrs, timeout=30)
            r.raise_for_status()
            omap = order_by_stmt[stmt_type]
            for tag in re.findall(r"defref_us-gaap_(\w+)", r.text):
                if tag not in omap:
                    omap[tag] = len(omap)
        except:
            pass

    print(f"✅ Presentation order from filing R-files: "
          f"IS={len(is_o)}  BS={len(bs_o)}  CFS={len(cfs_o)} concepts mapped")
    return is_o, bs_o, cfs_o

is_order, bs_order, cfs_order = get_filing_concept_order(
    CIK, tenk_filings[-1]["accessionNumber"])
ORDER_MAPS = {"IS": is_order, "BS": bs_order, "CFS": cfs_order}

# ── Build DataFrames sorted by filing presentation order ──────────────────────
all_facts = harvest_all_facts(facts_gaap, FY_ENDS)

by_stmt = {"IS": {}, "BS": {}, "CFS": {}}
for tag, info in all_facts.items():
    by_stmt[info["stmt"]][tag] = info

def stmt_to_df(stmt_facts, stmt_type):
    omap = ORDER_MAPS[stmt_type]
    def _key(item):
        # Filing order first; concepts not in R-files (supplemental) go to end alphabetically
        return (omap.get(item[0], 9999), item[1]["label"].lower())
    rows = []
    for tag, info in sorted(stmt_facts.items(), key=_key):
        row = {"Label": info["label"], "XBRL Concept": tag}
        for i, fy in enumerate(FY_LABELS):
            row[fy] = info["values"].get(FY_ENDS[i])
        rows.append(row)
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=["Label", "XBRL Concept"] + FY_LABELS)

df_is_full  = stmt_to_df(by_stmt["IS"],  "IS")
df_bs_full  = stmt_to_df(by_stmt["BS"],  "BS")
df_cfs_full = stmt_to_df(by_stmt["CFS"], "CFS")

print(f"✅ Harvested {len(all_facts)} us-gaap concepts with 10-K data:")
print(f"   Income Statement  : {len(by_stmt['IS']):>4} items")
print(f"   Balance Sheet     : {len(by_stmt['BS']):>4} items")
print(f"   Cash Flow Stmt    : {len(by_stmt['CFS']):>4} items")

✅ Presentation order from filing R-files: IS=0  BS=0  CFS=0 concepts mapped
✅ Harvested 270 us-gaap concepts with 10-K data:
   Income Statement  :  115 items
   Balance Sheet     :  125 items
   Cash Flow Stmt    :   30 items


In [6]:
# ── Cell 6: Excel Workbook Builder ───────────────────────────────────────────

# Style constants
NAVY       = "1F3864"
MED_GRAY   = "BFBFBF"
LIGHT_GRAY = "F2F2F2"
ALT_ROW    = "FAFAFA"
PROJ_FILL  = "EBF3FB"
PRIORITY_F = "FFF8E7"    # amber for key analyst rows
WACC_INPUT = "FFF2CC"
WACC_RSLT  = "D9EAD3"

THIN_TOP = Border(top=Side(style="thin"))

FMT_DOLLAR = '$#,##0.0;($#,##0.0);"-"'
FMT_PCT    = '0.0%;(0.0%);"-"'
FMT_EPS    = '$#,##0.00;($#,##0.00);"-"'
FMT_SHARES = '#,##0.0;(#,##0.0);"-"'
FMT_2DP    = '#,##0.00'

N_FY   = len(FY_LABELS)
N_PROJ = len(PROJ_LABELS)

# Column layout: A=spacer | B=Label | C=XBRL Tag | D+=FY data | then forecast
FS_LABEL_COL = 2
FS_TAG_COL   = 3
FS_DATA_COL  = 4
FS_PROJ_COL  = FS_DATA_COL + N_FY

def col(n): return get_column_letter(n)

def _unit_fmt(unit):
    if unit == "shares": return FMT_SHARES
    if unit == "pure":   return FMT_2DP
    return FMT_DOLLAR

def set_fs_widths(ws):
    ws.column_dimensions["A"].width = 3
    ws.column_dimensions[col(FS_LABEL_COL)].width = 52
    ws.column_dimensions[col(FS_TAG_COL)].width   = 56
    for i in range(N_FY):   ws.column_dimensions[col(FS_DATA_COL + i)].width = 13
    for i in range(N_PROJ): ws.column_dimensions[col(FS_PROJ_COL + i)].width = 13

def set_fs_header(ws, company, stmt_name, n_items):
    last_col = col(FS_PROJ_COL + N_PROJ - 1)
    ws.row_dimensions[1].height = 8
    ws.merge_cells(f"B2:{last_col}2")
    c = ws["B2"]
    c.value     = f"{company}  |  {stmt_name}"
    c.font      = Font(bold=True, size=14, color="FFFFFF", name="Calibri")
    c.fill      = PatternFill("solid", fgColor=NAVY)
    c.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[2].height = 24
    ws["B3"].value = (f"{n_items} XBRL concepts  •  All figures in $MM unless noted  "
                      f"•  Amber rows = key analyst items  •  Blue = actuals  "
                      f"•  Light-blue cols = analyst forecast input")
    ws["B3"].font  = Font(italic=True, size=9, color="808080", name="Calibri")
    ws.row_dimensions[3].height = 14
    hdr_fill = PatternFill("solid", fgColor=MED_GRAY)
    hdr_font = Font(bold=True, size=10, name="Calibri")
    hdr_aln  = Alignment(horizontal="center")
    for cn, txt in [(FS_LABEL_COL, "Line Item / Concept"), (FS_TAG_COL, "XBRL Tag (us-gaap)")]:
        c = ws.cell(row=4, column=cn, value=txt)
        c.font = hdr_font; c.fill = hdr_fill; c.alignment = hdr_aln
    for i, fy in enumerate(FY_LABELS):
        c = ws.cell(row=4, column=FS_DATA_COL+i, value=fy)
        c.font = hdr_font; c.fill = hdr_fill; c.alignment = hdr_aln
    proj_fill = PatternFill("solid", fgColor="D9D9D9")
    for i, pl in enumerate(PROJ_LABELS):
        c = ws.cell(row=4, column=FS_PROJ_COL+i, value=pl)
        c.font = Font(bold=True, size=10, name="Calibri", color="595959")
        c.fill = proj_fill; c.alignment = hdr_aln
    ws.auto_filter.ref = f"B4:{last_col}4"
    ws.freeze_panes = col(FS_DATA_COL) + "5"

def write_full_rows(ws, df, stmt_facts, start_row=5):
    r = start_row
    for _, row_data in df.iterrows():
        tag   = row_data.get("XBRL Concept", "")
        label = row_data.get("Label", tag)
        info  = stmt_facts.get(tag, {})
        unit  = info.get("unit", "USD")
        is_pr = info.get("priority", False)
        bg    = PRIORITY_F if is_pr else (ALT_ROW if (r - start_row) % 2 else "FFFFFF")
        rfill = PatternFill("solid", fgColor=bg)
        fmt   = _unit_fmt(unit)
        lc = ws.cell(row=r, column=FS_LABEL_COL, value=label)
        lc.font = Font(bold=is_pr, size=10, name="Calibri"); lc.fill = rfill
        tc = ws.cell(row=r, column=FS_TAG_COL, value=tag)
        tc.font = Font(size=9, name="Calibri", color="595959"); tc.fill = rfill
        for i, fy in enumerate(FY_LABELS):
            v = row_data.get(fy)
            has_val = pd.notna(v) and v is not None
            dc = ws.cell(row=r, column=FS_DATA_COL+i, value=float(v) if has_val else None)
            dc.number_format = fmt
            dc.font      = Font(color="0000FF" if has_val else "C0C0C0",
                                name="Calibri", size=10, bold=is_pr)
            dc.fill      = rfill
            dc.alignment = Alignment(horizontal="right")
        pfill = PatternFill("solid", fgColor=PROJ_FILL)
        for i in range(N_PROJ):
            pc = ws.cell(row=r, column=FS_PROJ_COL+i)
            pc.number_format = fmt
            pc.font = Font(name="Calibri", size=10); pc.fill = pfill
            pc.alignment = Alignment(horizontal="right")
        ws.row_dimensions[r].height = 15
        r += 1
    return r

# ─────────────────────────────────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

# ── COVER SHEET ───────────────────────────────────────────────────────────────
cover = wb.create_sheet("Cover")
cover.column_dimensions["A"].width = 4
cover.column_dimensions["B"].width = 34
cover.column_dimensions["C"].width = 46
cover.merge_cells("B2:D2")
t = cover["B2"]
t.value     = f"{company_name}  |  SEC 10-K  —  Full XBRL Financial Model"
t.font      = Font(bold=True, size=16, color="FFFFFF", name="Calibri")
t.fill      = PatternFill("solid", fgColor=NAVY)
t.alignment = Alignment(horizontal="left", vertical="center")
cover.row_dimensions[2].height = 30
for i, (k, v) in enumerate([
    ("Company",          company_name),
    ("Ticker",           TICKER.upper()),
    ("CIK",              CIK),
    ("Date Pulled",      TODAY),
    ("Fiscal Years",     ", ".join(FY_LABELS)),
    ("Forecast Columns", ", ".join(PROJ_LABELS)),
    ("Units",            "All values in $MM unless per-share or ratio"),
    ("Data Source",      "https://data.sec.gov  (XBRL company facts)"),
    ("IS Concepts",      str(len(by_stmt["IS"]))),
    ("BS Concepts",      str(len(by_stmt["BS"]))),
    ("CFS Concepts",     str(len(by_stmt["CFS"]))),
    ("Total Harvested",  str(len(all_facts))),
], start=4):
    cover.cell(row=i, column=2, value=k).font = Font(bold=True, name="Calibri", size=10)
    cover.cell(row=i, column=3, value=v).font = Font(name="Calibri", size=10)
cover.cell(row=17, column=2, value="Contents").font = Font(bold=True, size=11, name="Calibri")
for i, sname in enumerate(["Income Statement", "Balance Sheet",
                            "Cash Flow Statement", "WACC", "Tag Catalog"], start=18):
    c = cover.cell(row=i, column=2, value=sname)
    c.hyperlink = f"#{sname}!A1"
    c.font = Font(color="0563C1", underline="single", name="Calibri", size=10)

# ── INCOME STATEMENT ──────────────────────────────────────────────────────────
ws_is = wb.create_sheet("Income Statement")
set_fs_widths(ws_is)
set_fs_header(ws_is, company_name, "Income Statement", len(by_stmt["IS"]))
write_full_rows(ws_is, df_is_full, by_stmt["IS"])

# ── BALANCE SHEET ─────────────────────────────────────────────────────────────
ws_bs = wb.create_sheet("Balance Sheet")
set_fs_widths(ws_bs)
set_fs_header(ws_bs, company_name, "Balance Sheet", len(by_stmt["BS"]))
write_full_rows(ws_bs, df_bs_full, by_stmt["BS"])

# ── CASH FLOW STATEMENT ───────────────────────────────────────────────────────
ws_cfs = wb.create_sheet("Cash Flow Statement")
set_fs_widths(ws_cfs)
set_fs_header(ws_cfs, company_name, "Cash Flow Statement", len(by_stmt["CFS"]))
write_full_rows(ws_cfs, df_cfs_full, by_stmt["CFS"])

# ── WACC SHEET ────────────────────────────────────────────────────────────────
ws_w = wb.create_sheet("WACC")
ws_w.column_dimensions["A"].width = 4
ws_w.column_dimensions["B"].width = 38
ws_w.column_dimensions["C"].width = 16
ws_w.column_dimensions["D"].width = 16
ws_w.column_dimensions["E"].width = 13
ws_w.column_dimensions["F"].width = 13
ws_w.column_dimensions["G"].width = 13

def w_title(r, text):
    ws_w.merge_cells(f"B{r}:G{r}")
    c = ws_w[f"B{r}"]
    c.value = text
    c.font  = Font(bold=True, size=14, color="FFFFFF", name="Calibri")
    c.fill  = PatternFill("solid", fgColor=NAVY)
    c.alignment = Alignment(horizontal="left", vertical="center")
    ws_w.row_dimensions[r].height = 26

def w_sec(r, text):
    ws_w.merge_cells(f"B{r}:G{r}")
    c = ws_w[f"B{r}"]
    c.value = text
    c.font  = Font(bold=True, size=10, color="FFFFFF", name="Calibri")
    c.fill  = PatternFill("solid", fgColor="2E5F8A")
    ws_w.row_dimensions[r].height = 16

def w_row(r, label, value=None, formula=None, fmt=FMT_PCT, is_input=False, is_result=False):
    lc = ws_w.cell(row=r, column=2, value=label)
    lc.font = Font(bold=is_result, size=10, name="Calibri")
    if is_result: lc.fill = PatternFill("solid", fgColor=WACC_RSLT)
    vc = ws_w.cell(row=r, column=3, value=formula if formula else value)
    vc.number_format = fmt
    vc.alignment = Alignment(horizontal="center")
    if is_input:
        vc.fill = PatternFill("solid", fgColor=WACC_INPUT)
        vc.font = Font(color="0000FF", size=10, name="Calibri")
    elif is_result:
        vc.fill = PatternFill("solid", fgColor=WACC_RSLT)
        vc.font = Font(bold=True, color="375623", size=11, name="Calibri")
    else:
        vc.font = Font(color="000000", size=10, name="Calibri")
    ws_w.row_dimensions[r].height = 15

w_title(2, f"{company_name}  |  WACC Buildup")
ws_w["B3"].value = "Yellow cells = user inputs.  Green cell = WACC result (referenced by DCF model)."
ws_w["B3"].font  = Font(italic=True, size=9, color="808080", name="Calibri")

w_sec(5,  "COST OF DEBT")
w_row(6,  "Pre-tax cost of debt  (Kd)",            value=0.045, fmt=FMT_PCT, is_input=True)
w_row(7,  "Marginal tax rate  (T)",                 value=0.21,  fmt=FMT_PCT, is_input=True)
w_row(8,  "After-tax cost of debt  Kd × (1 − T)",  formula="=C6*(1-C7)", fmt=FMT_PCT)

w_sec(10, "COST OF EQUITY  —  CAPM")
w_row(11, "Risk-free rate  (Rf)  — 10-yr UST",     value=0.043,  fmt=FMT_PCT, is_input=True)
w_row(12, "Equity beta  (β)",                       value=1.20,   fmt=FMT_2DP, is_input=True)
w_row(13, "Equity risk premium  (ERP)",              value=0.055,  fmt=FMT_PCT, is_input=True)
w_row(14, "Cost of equity  Ke = Rf + β × ERP",      formula="=C11+C12*C13", fmt=FMT_PCT)

w_sec(16, "CAPITAL STRUCTURE")
for cn, txt in [(2,"Component"),(3,"Market Value ($MM)"),(4,"Target Override"),(5,"Weight")]:
    c = ws_w.cell(row=17, column=cn, value=txt)
    c.font = Font(bold=True, size=9, name="Calibri")
    c.fill = PatternFill("solid", fgColor=MED_GRAY)
    c.alignment = Alignment(horizontal="center")
ws_w.row_dimensions[17].height = 14
for r_, lbl, we_formula in [
    (18, "Equity (Market Capitalization)",  "=IF(D18<>\"\",D18,C18/(C18+C19))"),
    (19, "Net Debt  (Total Debt − Cash)",   "=1-E18"),
]:
    ws_w.cell(row=r_, column=2, value=lbl).font = Font(size=10, name="Calibri")
    inp = ws_w.cell(row=r_, column=3)
    inp.fill = PatternFill("solid", fgColor=WACC_INPUT)
    inp.font = Font(color="0000FF", size=10, name="Calibri")
    inp.number_format = FMT_DOLLAR
    inp.alignment = Alignment(horizontal="center")
    we = ws_w.cell(row=r_, column=5, value=we_formula)
    we.number_format = FMT_PCT
    we.font = Font(color="000000", size=10, name="Calibri")
    we.alignment = Alignment(horizontal="center")
    ws_w.row_dimensions[r_].height = 15

w_sec(21, "WACC")
ws_w.cell(row=22, column=2, value="WACC  =  Ke × We  +  Kd(1−T) × Wd").font = Font(bold=True, size=11, name="Calibri")
ws_w.cell(row=22, column=2).fill = PatternFill("solid", fgColor=WACC_RSLT)
wr = ws_w.cell(row=22, column=3, value="=C14*E18+C8*E19")
wr.number_format = FMT_PCT
wr.font      = Font(bold=True, size=13, color="375623", name="Calibri")
wr.fill      = PatternFill("solid", fgColor=WACC_RSLT)
wr.alignment = Alignment(horizontal="center", vertical="center")
ws_w.row_dimensions[22].height = 22

w_sec(25, "SENSITIVITY  —  WACC  vs  Beta  &  Equity Risk Premium")
ws_w.cell(row=26, column=2, value="WACC").font  = Font(bold=True, size=9, name="Calibri")
ws_w.cell(row=26, column=3, value="Beta →").font = Font(bold=True, size=9, name="Calibri", color="595959")
ws_w.cell(row=27, column=2, value="ERP ↓").font  = Font(bold=True, size=9, name="Calibri", color="595959")

BETA_OFFSETS = [-0.30, -0.15, 0.00, 0.15, 0.30]
ERP_OFFSETS  = [-0.010, -0.005, 0.000, 0.005, 0.010]

for j, bo in enumerate(BETA_OFFSETS):
    beta_f = "=C12" if bo == 0 else f"={bo:+.2f}+C12"
    c = ws_w.cell(row=26, column=3+j, value=beta_f)
    c.number_format = FMT_2DP
    c.font = Font(bold=True, size=9, name="Calibri")
    c.fill = PatternFill("solid", fgColor=MED_GRAY)
    c.alignment = Alignment(horizontal="center")

for i, eo in enumerate(ERP_OFFSETS):
    erp_f = "=C13" if eo == 0 else f"={eo:+.3f}+C13"
    er = ws_w.cell(row=27+i, column=2, value=erp_f)
    er.number_format = FMT_PCT
    er.font = Font(bold=True, size=9, name="Calibri")
    er.fill = PatternFill("solid", fgColor=MED_GRAY)
    er.alignment = Alignment(horizontal="center")
    for j, bo in enumerate(BETA_OFFSETS):
        beta_expr = "C12" if bo == 0 else f"({bo:+.2f}+C12)"
        erp_expr  = "C13" if eo == 0 else f"({eo:+.3f}+C13)"
        f = f"=(C11+{beta_expr}*{erp_expr})*E18+C8*E19"
        vc = ws_w.cell(row=27+i, column=3+j, value=f)
        vc.number_format = FMT_PCT
        vc.font = Font(size=9, name="Calibri")
        vc.alignment = Alignment(horizontal="center")
        if i == 2 and j == 2:
            vc.fill = PatternFill("solid", fgColor=WACC_RSLT)
            vc.font = Font(bold=True, size=9, color="375623", name="Calibri")
    ws_w.row_dimensions[27+i].height = 14

ws_w.cell(row=34, column=2, value="Notes").font = Font(bold=True, size=9, name="Calibri")
for i, note in enumerate([
    "• Yellow cells are user inputs. Update Rf, β, ERP, market cap, and net debt for this company.",
    "• Risk-free rate: current 10-year US Treasury yield.",
    "• ERP: Damodaran estimates ~5.5% for US equities (implied ERP, Jan 2025).",
    "• Beta: observed 5-year monthly beta vs. S&P 500 (Bloomberg, Yahoo Finance).",
    "• Market cap and net debt: pull from the Balance Sheet tab or live market data.",
    "• Cell C22 (WACC result) can be referenced directly in a DCF model.",
], start=35):
    ws_w.cell(row=i, column=2, value=note).font = Font(size=9, name="Calibri", color="595959")
    ws_w.row_dimensions[i].height = 13

# ── TAG CATALOG ───────────────────────────────────────────────────────────────
ws_cat = wb.create_sheet("Tag Catalog")
ws_cat.freeze_panes = "A2"
ws_cat.append(["Statement", "Key", "XBRL Tag (us-gaap)", "Label", "Period Type", "Unit"] + FY_LABELS)
for tag, info in sorted(all_facts.items(), key=lambda x: (x[1]["stmt"], x[1]["label"].lower())):
    ws_cat.append([
        info["stmt"],
        "★" if info["priority"] else "",
        tag,
        info["label"],
        info["period_type"],
        info["unit"],
    ] + [info["values"].get(fe) for fe in FY_ENDS])
n_cat = len(all_facts) + 1
tbl_cat = Table(displayName="TagCatalog", ref=f"A1:{col(6+N_FY)}{n_cat}")
tbl_cat.tableStyleInfo = TableStyleInfo(name="TableStyleMedium9", showRowStripes=True)
ws_cat.add_table(tbl_cat)
for cn, w in [(1,14),(2,6),(3,58),(4,52),(5,12),(6,8)] + [(6+i+1,13) for i in range(N_FY)]:
    ws_cat.column_dimensions[col(cn)].width = w

print("✅ Excel workbook built successfully")
print(f"   Income Statement  : {len(by_stmt['IS'])} rows")
print(f"   Balance Sheet     : {len(by_stmt['BS'])} rows")
print(f"   Cash Flow Stmt    : {len(by_stmt['CFS'])} rows")
print(f"   Tag Catalog       : {len(all_facts)} concepts")

✅ Excel workbook built successfully
   Income Statement  : 115 rows
   Balance Sheet     : 125 rows
   Cash Flow Stmt    : 30 rows
   Tag Catalog       : 270 concepts


In [7]:
# ── Cell 7: Save & Summary ────────────────────────────────────────────────────
wb.save(OUTPUT_FILE)
print(f"✅ Saved: {OUTPUT_FILE}")
print()
print("Summary")
print("-------")
print(f"  Company          : {company_name}")
print(f"  Ticker           : {TICKER.upper()}")
print(f"  CIK              : {CIK}")
print(f"  Fiscal Years     : {', '.join(FY_LABELS)}")
print(f"  Forecast Columns : {', '.join(PROJ_LABELS)}")
print(f"  XBRL concepts    : {len(all_facts)}")
print(f"    IS concepts    : {len(by_stmt['IS'])}")
print(f"    BS concepts    : {len(by_stmt['BS'])}")
print(f"    CFS concepts   : {len(by_stmt['CFS'])}")
print(f"  Output file      : {OUTPUT_FILE}")

✅ Saved: OKE_10K_Historical_2026-04-18.xlsx

Summary
-------
  Company          : ONEOK INC /NEW/
  Ticker           : OKE
  CIK              : 0001039684
  Fiscal Years     : FY2021, FY2022, FY2023, FY2024, FY2025
  Forecast Columns : FY2026E, FY2027E, FY2028E
  XBRL concepts    : 270
    IS concepts    : 115
    BS concepts    : 125
    CFS concepts   : 30
  Output file      : OKE_10K_Historical_2026-04-18.xlsx
